In [10]:
print("s;fmgdfs")

s;fmgdfs


In [11]:
%pwd
import os
os.chdir("D:\mlops\Wine-Quality-Data-Science-Projects-MLOPS-")
%pwd

'D:\\mlops\\Wine-Quality-Data-Science-Projects-MLOPS-'

In [22]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelTrainerConfig:
    root_dir: Path
    train_path: Path
    test_path: Path
    model_name: Path
    
    alpha: float
    l1_ratio: float
    target_column:str

In [23]:
from src.DATA_SCIENCE_Project.constants import *
from src.DATA_SCIENCE_Project.utils.helper import read_yaml,create_directories

class ConfigurationManager:
    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH
                 ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        
        create_directories([self.config.artifacts_root])
        
    def get_model_trainer_config(self)-> ModelTrainerConfig:
        config =  self.config.model_trainer
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN
        create_directories([config.root_dir])
        model_train_config  = ModelTrainerConfig(
            root_dir=config.root_dir,
            train_path=config.train_path,
            test_path=config.test_path,
            model_name = config.model_name,
            alpha = params.alpha,
            l1_ratio = params.l1_ratio,
            target_column = schema.name
        )
        return model_train_config


In [24]:
import os
from src.DATA_SCIENCE_Project import logger
from sklearn.model_selection import train_test_split
from sklearn.linear_model import ElasticNet
import pandas as pd
import joblib


In [27]:
class ModelTrainerComponent:
        def __init__(self,config:ModelTrainerConfig):
            self.config = config
        
        def train(self):
            
            # 1. Load the data properly
            train_data = pd.read_csv(self.config.train_path)
            test_data = pd.read_csv(self.config.test_path)

            # 2. Split Features (X) and Target (y)
            tar = self.config.target_column
            print(tar)
            train_x = train_data.drop([tar], axis=1)
            test_x = test_data.drop([tar], axis=1)
            
            train_y = train_data[[tar]]
            test_y = test_data[[tar]]

            # 3. Model Initialization and Training
            # Fixed typos: alpha and l1_ratio
            lr = ElasticNet(alpha=self.config.alpha, l1_ratio=self.config.l1_ratio, random_state=42)
            lr.fit(train_x, train_y)

            # 4. Save the Model
            model_path = os.path.join(self.config.root_dir, self.config.model_name)
            joblib.dump(lr, model_path)
            
        
    

In [28]:
try:
    cfm = ConfigurationManager()
    model_trainer_manager = cfm.get_model_trainer_config()
    model_trainer_component = ModelTrainerComponent(model_trainer_manager)
    model_trainer_component.train()
    logger.info("Pipeline Ran Sucesssfully ✅😭 ")
except Exception as e:
    logger.error("Are bhia kuch Error Aaa Gaya hai Yaar....")
    logger.error(e)
    raise e

[24-04-2026 08:03:33 PM - helper - INFO - yaml file: config\config.yaml Loaded Sucessfully]
[24-04-2026 08:03:33 PM - helper - INFO - yaml file: params.yaml Loaded Sucessfully]
[24-04-2026 08:03:33 PM - helper - INFO - yaml file: schema.yaml Loaded Sucessfully]
[24-04-2026 08:03:33 PM - helper - INFO - Created Directory of path : artifacts]
[24-04-2026 08:03:33 PM - helper - INFO - Created Directory of path : artifacts/model_trainer]
quality
[24-04-2026 08:03:33 PM - 1810912711 - INFO - Pipeline Ran Sucesssfully ✅😭 ]
